# Examples

Build a crystal structure in the Simple representation: a cell, sites in
reduced coordinates, OPTIMADE-style species, and the species occupying each site.

In [1]:
from httk.atomistic import Structure, StructurePrimitiveView

structure = Structure(
    cell=[[5.64, 0.0, 0.0], [0.0, 5.64, 0.0], [0.0, 0.0, 5.64]],
    sites=[[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]],
    species=[
        {"name": "Na", "chemical_symbols": ["Na"], "concentration": [1.0]},
        {"name": "Cl", "chemical_symbols": ["Cl"], "concentration": [1.0]},
    ],
    species_at_sites=["Na", "Cl"],
)

print("Species:", [s.name for s in structure.species])
print("Sites:", len(structure.sites))

Species: ['Na', 'Cl']
Sites: 2


A view presents the same structure through another representation. The primitive
view is a genuine spglib-like `(lattice, positions, numbers)` tuple:

In [2]:
lattice, positions, numbers = StructurePrimitiveView(structure)
print("Atomic numbers:", numbers)

Atomic numbers: (11, 17)


The `cell`, `sites`, and `species` components get the same view/backend treatment.
`structure.cell` is a `Cell` with derived quantities, and `SpeciesPrimitiveView`
presents a species as an OPTIMADE dict:

In [3]:
from httk.atomistic import SpeciesPrimitiveView

cell = structure.cell
# lengths/volume are exact (SurdScalar); render them to floats at the presentation boundary.
print("Cell lengths:", tuple(round(length.to_float(), 4) for length in cell.lengths))
print("Cell volume:", cell.volume.to_float())
print("Species dict:", dict(SpeciesPrimitiveView(structure.species[0])))

Cell lengths: (5.64, 5.64, 5.64)
Cell volume: 179.40614399999998
Species dict: {'name': 'Na', 'chemical_symbols': ['Na'], 'concentration': [1.0]}


Species that are not single, unattached chemical elements (for example a site with
a vacancy probability) cannot be expressed as bare atomic numbers, so the
primitive view refuses them:

In [4]:
vacancy_structure = Structure(
    cell=[[3.0, 0.0, 0.0], [0.0, 3.0, 0.0], [0.0, 0.0, 3.0]],
    sites=[[0.0, 0.0, 0.0]],
    species=[{"name": "Ti", "chemical_symbols": ["Ti", "vacancy"], "concentration": [0.9, 0.1]}],
    species_at_sites=["Ti"],
)
try:
    StructurePrimitiveView(vacancy_structure)
except TypeError as exc:
    print("TypeError:", exc)

TypeError: This structure cannot be represented as a primitive structure (species 'Ti' is not a single, unattached chemical element)
